In [ ]:
# pip install requests json python-binance pandas openpyxl tqdm
# python = 3.12

In [ ]:
# 관련 docs
# [01]. binance api
#       https://developers.binance.com/docs/binance-spot-api-docs

In [1]:
import requests
import json
from binance.client import Client

import pandas as pd
import openpyxl

import os
import time

from tqdm import tqdm

In [2]:
# 주의
KEYS = pd.read_excel("private_keys.xlsx")
api_key = KEYS['API Key'][0]
api_secret = KEYS['Secret Key'][0]

client = Client(api_key, api_secret)


In [3]:
KLINE_COLUMNS = [
    'Open time', 'Open', 'High', 'Low', 'Close', 'Volume', 
    'Close time', 'Quote asset volume', 'Number of trades', 
    'Taker buy base asset volume', 'Taker buy quote asset volume', 'Ignore'
]

class BinanceDataCollector:
    """
    바이낸스 API를 사용하여 심볼 정보 및 캔들스틱 데이터를 수집하는 클래스입니다.
    """
    def __init__(self, api_key, api_secret):
        # binance.client.Client 객체를 인스턴스 변수로 저장
        self.client = Client(api_key, api_secret)
        self.all_symbols = self.get_all_symbols()

    def get_all_symbols(self):
        """
        Binance 상장된 모든 Symbol 추출목적
        """
        info = self.client.get_exchange_info() # 심볼 + 필터
        
        symbols = [x['symbol'] for x in info['symbols']]
        print(f"전체 심볼 개수 : {len(symbols)}")
        return symbols

    def get_binance_klines(self, symbol, interval, start_str, end_str=None):
        # start_str과 end_str은 날짜 문자열 (예: '1 Jan, 2024').

        """
        Binance API에서 캔들스틱 데이터 추출
        , API 과다호출로 인한 ban을 받기 위해서 1회 호출시 symbol개수 한정
        """
        try:
            klines = self.client.get_historical_klines(
                symbol, 
                interval, 
                start_str, 
                end_str,
                limit = 2000 # 한 번에 500개 symbol만 호출
            )
            if not klines:
                print(f"{symbol} : 심볼 데이터 없음. 이유는 체크필요.")
                return None
            
            # DataFrame으로 변환하고 컬럼 이름 명시적으로 설정
            data = pd.DataFrame(klines, columns=KLINE_COLUMNS)

            # 데이터 타입 변환 및 인덱스 설정 (기존 get_binance_klines 로직 통합)
            data['Open time'] = pd.to_datetime(data['Open time'], unit='ms')
            data['Close time'] = pd.to_datetime(data['Close time'], unit='ms')
            
            numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
            for col in numeric_cols:
                # 숫자형 컬럼을 float으로 변환
                data[col] = pd.to_numeric(data[col])
                
            data = data.set_index('Open time')
            
            print(f"{symbol} : ({len(data)}개) 데이터 추출 완료")
            return data
        
        except Exception as e:
            print(f"{symbol} : 데이터 추출 오류 코드({e})")
            return None
        finally:
            time.sleep(1) # ban 방지

In [4]:
data_collector = BinanceDataCollector(api_key, api_secret)

# INTERVAL = Client.KLINE_INTERVAL_1HOUR
INTERVAL = Client.KLINE_INTERVAL_1DAY

START_DATE = "1 May, 2024"

전체 심볼 개수 : 3534


In [5]:
# 데이터 dict형식으로 받을 예정
all_klines_data = {}

print("==========Data Extract Process is started.==========")

df_list =[]

# 인스턴스 변수에 저장된 전체 심볼 목록 사용
all_symbols = data_collector.all_symbols

# test
# for symbol in tqdm(all_symbols,miniters=100, desc="처리 중"):
for symbol in tqdm(all_symbols,miniters=100, desc="처리 중"):
    df_klines = data_collector.get_binance_klines(symbol, INTERVAL, START_DATE)
    
    if df_klines is not None:
        df_klines['Symbol'] = symbol
        # all_klines_data[symbol] = df_klines
        df_list.append(df_klines)

if df_list:
    df_final = pd.concat(df_list)
    df_final = df_final.sort_values(by = ['Symbol', 'Open time'])
    print("==========Data Extract Process is done.==========")
    print(f"총 수집된 행(Row) 수: {len(df_final)}")
    print(df_final.head())
else :
    print('no data')

==========Data Extract Process is started.==========


처리 중:   0%|          | 0/3534 [00:00<?, ?it/s]

ETHBTC : (684개) 데이터 추출 완료
LTCBTC : (684개) 데이터 추출 완료
BNBBTC : (684개) 데이터 추출 완료
NEOBTC : (684개) 데이터 추출 완료
QTUMETH : (644개) 데이터 추출 완료
EOSETH : (391개) 데이터 추출 완료
SNTETH : (351개) 데이터 추출 완료
BNTETH : (234개) 데이터 추출 완료
BCCBTC : 심볼 데이터 없음. 이유는 체크필요.


처리 중:   0%|          | 9/3534 [00:10<1:06:38,  1.13s/it]

GASBTC : (684개) 데이터 추출 완료


...

MBLUSDT : (684개) 데이터 추출 완료


처리 중:  21%|██        | 750/3534 [14:05<52:01,  1.12s/it]

COTIBNB : 심볼 데이터 없음. 이유는 체크필요.


처리 중:  21%|██        | 750/3534 [14:06<52:23,  1.13s/it]


KeyboardInterrupt: 

In [10]:
len(df_list)

250

In [12]:
df_all = pd.DataFrame()
for df_tmp in df_list:
    df_all = pd.concat([df_all, df_tmp],ignore_index=True)

In [19]:
import pyarrow

In [23]:
df_all.to_parquet('./data/df_260315.parquet',engine='fastparquet')